# 02. Univariate GARCH Models

The first notebook showed that volatility clustering is present in the daily market series, especially around crisis periods. This second notebook takes the next natural step: we model conditional volatility directly. The economic question is whether volatility became more persistent, more asymmetric or more heavy-tailed after COVID-19.

We focus on four core assets that capture equity, credit, commodity and safe-haven risk:
- S&P 500,
- US HY Bonds,
- Oil futures,
- Gold.

## 1. Setup

We reuse the exact same cleaned daily return construction as in notebook 01. This is important because any difference between the pre- and post-COVID estimates should come from the data-generating process itself, not from a change in preprocessing choices.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from project2_garch_utils import (
    GARCH_ASSETS,
    GARCH_LABELS,
    MODEL_SPECS,
    load_garch_samples,
    estimate_all_garch_models,
    build_comparison_table,
    plot_conditional_volatility,
    save_garch_outputs,
)
from project2_data_utils import ensure_output_dirs, save_figure

sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = "{:.4f}".format
ensure_output_dirs()

## 2. Load the analysis sample

We rebuild the aligned daily return sample from the raw Excel workbook and keep the same pre/post split used in notebook 01. This ensures that the volatility models are estimated on exactly the same empirical objects as the stylized facts.

In [ ]:
aligned_returns, pre_covid, post_covid = load_garch_samples()

print(f"Full aligned sample: {aligned_returns.shape}")
print(f"Pre-COVID sample:    {pre_covid.shape}")
print(f"Post-COVID sample:   {post_covid.shape}")

## 3. Model design

We estimate three standard models from the course for each asset and each sub-sample:
- **GARCH(1,1)** as the baseline volatility model;
- **GJR-GARCH** to test for leverage or asymmetry;
- **GARCH-t** to allow for heavy-tailed innovations.

This combination is useful because each model answers a different economic question. The baseline GARCH measures volatility persistence. The GJR-GARCH checks whether bad news changes volatility differently from good news. The Student-t specification checks whether the Gaussian assumption is too restrictive in the tails.

In [ ]:
model_design = pd.DataFrame({
    "model": list(MODEL_SPECS.keys()),
    "economic_role": [
        "baseline volatility persistence",
        "asymmetric volatility / leverage effect",
        "fat tails in the innovation distribution",
    ],
})
model_design

## 4. Estimate all univariate volatility models

We now estimate the full grid of models: 4 assets ? 2 sub-samples ? 3 specifications. The summary table extracts the parameters that matter most economically:
- `alpha_1` for short-run shock sensitivity,
- `beta_1` for volatility persistence,
- `gamma_1` for leverage in the GJR model,
- `nu` for tail thickness in the Student-t model.

We also save diagnostics because an estimated GARCH model is only useful if it removes most of the serial dependence left in volatility.

In [ ]:
garch_summary, fitted_models = estimate_all_garch_models(pre_covid, post_covid)
save_garch_outputs(garch_summary)

garch_summary.head(12)

## 5. Compact comparison table

The raw estimation output is long, so we condense it into a more readable table focused on persistence, asymmetry and residual diagnostics. This is the table we would most naturally interpret in the report.

In [ ]:
comparison_table = build_comparison_table(garch_summary)
comparison_table

## 6. Read the persistence and leverage patterns by asset

At this point, the most useful comparison is not model-by-model but asset-by-asset. We therefore pivot the main parameters into a format that makes the pre/post change easy to read.

In [ ]:
persistence_table = comparison_table.pivot_table(
    index=["asset_label", "model"],
    columns="sample",
    values="persistence",
)
persistence_table

In [ ]:
leverage_table = comparison_table.loc[comparison_table["model"] == "GJR-GARCH", [
    "asset_label", "sample", "gamma_1", "engle_ng_joint_pvalue"
]]
leverage_table

## 7. Residual diagnostics

Two diagnostics are especially important after fitting GARCH models:
- **Ljung-Box on standardized residuals**, to check whether mean dynamics remain in the filtered series;
- **Ljung-Box on squared standardized residuals**, to check whether conditional heteroskedasticity remains;
- **Engle-Ng**, to see whether asymmetry is still left unexplained.

If the p-values are comfortably above standard thresholds, the variance model is doing a reasonable job. If not, the volatility process may still be misspecified.

In [ ]:
diagnostics_table = comparison_table[[
    "asset_label",
    "sample",
    "model",
    "lb_resid_pvalue_lag10",
    "lb_sq_resid_pvalue_lag10",
    "engle_ng_joint_pvalue",
]].copy()
diagnostics_table

## 8. Conditional volatility plots

Tables are useful for parameter interpretation, but volatility models are easier to understand visually when we look at the estimated conditional standard deviation. We therefore plot the baseline GARCH(1,1) volatility for each core asset before and after COVID.

In [ ]:
for asset_name in GARCH_ASSETS:
    volatility_figure = plot_conditional_volatility(
        pre_series=pre_covid[asset_name],
        post_series=post_covid[asset_name],
        asset_name=asset_name,
    )
    save_figure(volatility_figure, f"02_conditional_volatility_{asset_name}.png")
    plt.show()

## 9. A simple ranking of post-COVID persistence

The most direct way to summarize this notebook is to rank post-COVID persistence across the four assets. Highly persistent post-COVID volatility suggests that shocks take longer to dissipate, which is one concrete way in which the structure of risk may have changed.

In [ ]:
post_persistence_ranking = (
    comparison_table.loc[comparison_table["sample"] == "Post-COVID", ["asset_label", "model", "persistence"]]
    .sort_values(["model", "persistence"], ascending=[True, False])
)
post_persistence_ranking

## 10. Main takeaways from notebook 02

This notebook tells us whether volatility persistence, leverage effects and heavy tails became more pronounced after COVID-19. These are core ingredients of the broader structure of risk. The next notebook will move from univariate volatility to cross-asset dependence through rolling correlations and DCC-GARCH.